
# Adult Rest/Movie Analysis: ISC and IDE Results

This notebook analyzes the adult rest/movie dataset including:
- ISC and TPHATE_DiffOp_IDE for aeronaut and mickey tasks
- TPHATE_DiffOp_IDE for rest (resting state)
- Task difference maps (rest - aeronaut and rest - mickey) with significance thresholding


In [ ]:

import numpy as np
import pandas as pd
import os, sys, glob
import matplotlib.pyplot as plt
import seaborn as sns
import nibabel as nib
from nilearn import plotting, image
from nilearn.maskers import NiftiMasker
import adult_restmovie_utils as aru
import adult_restmovie_config as arc
import plotting_helpers as helper
import stats_helpers as sh
from obspy.imaging.cm import viridis_white
%matplotlib inline
%load_ext autoreload
%autoreload 2
results_dir = aru.get_results_dir()


## 1. Load and visualize average ISC and IDE maps for aeronaut and mickey tasks


In [ ]:

# Load average results for aeronaut and mickey tasks

print(f"Results directory: {results_dir}")
results_dir = aru.get_results_dir()
# Define tasks and measures
movie_tasks = ['aeronaut' ]
measures = {'ISC': 'ISC', 'TPHATE_DiffOp_IDE': 'IDE'}

# Load the average result volumes
avg_results = {}
for task in movie_tasks:
    avg_results[task] = {}
    for measure in measures.keys():
        fn = f'{results_dir}/{task}_{measure}_average_results.nii.gz'
        if os.path.exists(fn):
            avg_results[task][measure] = nib.load(fn)
            print(f"Loaded {task} {measure}: {avg_results[task][measure].shape}; mean: {np.nanmean(avg_results[task][measure].get_fdata()):.3f}")
        else:
            print(f"Warning: {fn} not found")

# Prepare data for surface plotting grid
# We'll create a 2x2 grid: aeronaut ISC, aeronaut IDE, mickey ISC, mickey IDE
nifti_images = []
titles = []

for task in movie_tasks:
    for measure, label in measures.items():
        if measure in avg_results[task]:
            nifti_images.append(avg_results[task][measure])
            titles.append(f'{task.capitalize()} {label}')

# Determine colorbar ranges
# ISC typically ranges from 0 to ~0.5
# IDE typically ranges from ~2 to ~20
cbar_ranges = [(0, 0.4), (1, 18)]
cmaps = ['magma', viridis_white]
cbar_labels = ['ISC', 'IDE']

print(f"Prepared {len(nifti_images)} images for visualization")
print(f"Titles: {titles}")

# Create the surface plot grid for movie tasks (aeronaut and mickey)
output_path = os.path.join('main_plots/adult_aeronaut_ISC_IDE_surface_grid.pdf')

# # Generate individual surface plots
# temp_fns = []
# for idx, (img, title, cmap, cbar_range) in enumerate(zip(nifti_images, titles, cmaps, cbar_ranges)):
#     temp_fn = f'/tmp/adult_plot_{idx}.png'
    
#     helper.generate_surface_plot(
#         data_fn=img, 
#         image_fn=temp_fn,
#         atlas='searchlight', 
#         cmap=cmap, 
#         cbar_range=cbar_range,
#         surf_type='fslr', 
#         target_density='32k',
#         include_cbar=True, 
#         title=title,
#         method='linear', 
#         threshold=None, 
#         mask_medial_wall=True
#     )
#     temp_fns.append(temp_fn)

# # Compile all surface plots into a grid
# helper.compile_surface_plots_to_grid(
#     image_files=temp_fns,
#     atlas='searchlight',
#     data_files=nifti_images,
#     surf_type='fslr',
#     target_density='32k',
#     output_path=output_path,
#     main_title='Adult Aeronaut'
# )


In [ ]:
masker = NiftiMasker(mask_img=aru.get_intersect_mask())
masked = masker.fit_transform(avg_results["aeronaut"]["ISC"])
np.nanmean(masked), np.nanstd(masked), np.nanmax(masked), np.nanmin(masked)

## 2. Visualize TPHATE_DiffOp_IDE for rest (resting state)


In [ ]:


# Load rest task IDE results
rest_ide_fn = f'{results_dir}/rest_TPHATE_DiffOp_IDE_average_results.nii.gz'

if os.path.exists(rest_ide_fn):
    rest_ide_img = nib.load(rest_ide_fn)
    print(f"Loaded rest IDE: {rest_ide_img.shape}")
    print(f"Mean IDE value: {np.nanmean(rest_ide_img.get_fdata()):.3f}")
    
    # Generate surface plot for rest IDE
    output_fn = os.path.join('main_plots/rest_TPHATE_DiffOp_IDE_surface.pdf')
    
    helper.generate_surface_plot(
        data_fn=rest_ide_img,
        image_fn=output_fn,
        atlas='searchlight',
        cmap=viridis_white,
        cbar_range=(1, 25),
        surf_type='fslr',
        target_density='32k',
        include_cbar=True,
        title='Rest (Resting State) IDE',
        method='linear',
        threshold=None,
        mask_medial_wall=True
    )



## 3. Task Difference Maps: Rest - Aeronaut and Rest - Mickey

These maps show regions where IDE differs significantly between rest (resting state) and movie viewing tasks, with significance thresholding.


In [ ]:

# Load the task difference maps (rest - movie tasks)
diff_maps_dir = f'{results_dir}/task_difference_maps'
task_pairs = ['aeronaut_rest']

# Storage for difference maps
diff_results = {}

for pair in task_pairs:
    diff_results[pair] = {}
    
    # Load thresholded difference map
    thresh_fn = f'{diff_maps_dir}/{pair}_TPHATE_DiffOp_IDE_SL_difference_maps_avg_thresholded.nii.gz'
    if os.path.exists(thresh_fn):
        diff_results[pair]['thresholded'] = nib.load(thresh_fn)
        print(f"Loaded {pair} thresholded map: {diff_results[pair]['thresholded'].shape}")
    else:
        print(f"Warning: {thresh_fn} not found")
    
    # Load unthresholded difference map for reference
    unthresh_fn = f'{diff_maps_dir}/{pair}_TPHATE_DiffOp_IDE_SL_difference_maps_avg_unthresholded.nii.gz'
    if os.path.exists(unthresh_fn):
        diff_results[pair]['unthresholded'] = nib.load(unthresh_fn)
        print(f"Loaded {pair} unthresholded map: {diff_results[pair]['unthresholded'].shape}")

# Determine appropriate colorbar range for difference maps
# These are differences, so we want a diverging colormap centered at 0

# # Check the range of values in the thresholded maps
# diff_ranges = []
# for pair in task_pairs:
#     if 'thresholded' in diff_results[pair]:
#         data = diff_results[pair]['thresholded'].get_fdata()
#         # Remove NaN values for range calculation
#         valid_data = data[~np.isnan(data)]
#         if len(valid_data) > 0:
#             vmin, vmax = np.nanmin(valid_data), np.nanmax(valid_data)
#             diff_ranges.append((vmin, vmax))
#             print(f"{pair} range: {vmin:.3f} to {vmax:.3f}")

# # Use symmetric range for diverging colormap
# if diff_ranges:
#     abs_max = max(abs(r[0]) for r in diff_ranges + [(0, 0)] if r[0] is not None) 
#     abs_max = max(abs_max, max(abs(r[1]) for r in diff_ranges + [(0, 0)] if r[1] is not None))
#     cbar_range_diff = (-abs_max, abs_max)
#     print(f"Using symmetric colorbar range: {cbar_range_diff}")
# else:
#     cbar_range_diff = (-5, 5)
#     print("Using default colorbar range: (-5, 5)")

# Get diverging colormap
div_cmap = helper.diverging_colormap_bp()

# Visualize rest - aeronaut difference map (thresholded)
pair = 'aeronaut_rest'
if 'thresholded' in diff_results[pair]:
    output_fn = os.path.join(f'main_plots/{pair}_IDE_difference_thresholded_surface.pdf')
    
    # helper.generate_surface_plot(
    #     data_fn=diff_results[pair]['thresholded'],
    #     image_fn=output_fn,
    #     atlas='searchlight',
    #     cmap=div_cmap,
    #     cbar_range=[-10,10],
    #     surf_type='fslr',
    #     target_density='32k',
    #     include_cbar=True,
    #     title='Rest - Aeronaut IDE (thresholded)',
    #     method='linear',
    #     threshold=None,  # Already thresholded
    #     mask_medial_wall=True
    # )
    
    # # Display the plot
    # if os.path.exists(output_fn):
    #     img_array = plt.imread(output_fn)
    #     fig, ax = plt.subplots(figsize=(15, 8))
    #     ax.imshow(img_array)
    #     ax.axis('off')
    #     plt.tight_layout()
    #     plt.show()
    #     print(f"Saved to: {output_fn}")

# # Visualize rest - mickey difference map (thresholded)
# pair = 'mickey_rest'
# if 'thresholded' in diff_results[pair]:
#     output_fn = os.path.join(results_dir, f'{pair}_IDE_difference_thresholded_surface.png')
    
#     helper.generate_surface_plot(
#         data_fn=diff_results[pair]['thresholded'],
#         image_fn=output_fn,
#         atlas='searchlight',
#         cmap=div_cmap,
#         cbar_range=cbar_range_diff,
#         surf_type='fsaverage',
#         target_density='41k',
#         include_cbar=True,
#         title='Rest - Mickey IDE (thresholded)',
#         method='nearest',
#         threshold=None,  # Already thresholded
#         mask_medial_wall=True
#     )
    
#     # Display the plot
#     if os.path.exists(output_fn):
#         img_array = plt.imread(output_fn)
#         fig, ax = plt.subplots(figsize=(15, 8))
#         ax.imshow(img_array)
#         ax.axis('off')
#         plt.tight_layout()
#         plt.show()
#         print(f"Saved to: {output_fn}")

# # Create a combined figure showing both difference maps side by side
# fig, axes = plt.subplots(1, 2, figsize=(24, 10))

# for idx, pair in enumerate(task_pairs):
#     if 'thresholded' in diff_results[pair]:
#         temp_fn = f'/tmp/diff_plot_{pair}.png'
        
#         task_name = pair.split('_')[0].capitalize()
#         title = f'Rest - {task_name} IDE (thresholded)'
        
#         helper.generate_surface_plot(
#             data_fn=diff_results[pair]['thresholded'],
#             image_fn=temp_fn,
#             atlas='searchlight',
#             cmap=div_cmap,
#             cbar_range=cbar_range_diff,
#             surf_type='fsaverage',
#             target_density='41k',
#             include_cbar=True,
#             title=title,
#             method='nearest',
#             threshold=None,
#             mask_medial_wall=True
#         )
        
#         if os.path.exists(temp_fn):
#             img_array = plt.imread(temp_fn)
#             axes[idx].imshow(img_array)
#             axes[idx].axis('off')
#             axes[idx].set_title(title, fontsize=14, fontweight='bold')

# plt.suptitle('Task Differences: Resting State vs Movie Viewing (IDE)', 
#              fontsize=16, fontweight='bold', y=0.98)
# plt.tight_layout()

# output_combined = os.path.join(results_dir, 'rest_vs_movies_IDE_differences_combined.png')
# plt.savefig(output_combined, dpi=150, bbox_inches='tight')
# plt.show()
# print(f"Saved combined difference maps to: {output_combined}")



## 4. Summary Statistics

Let's examine some summary statistics about the differences between conditions.


In [ ]:

# Calculate summary statistics for each difference map
print("=" * 80)
print("SUMMARY STATISTICS FOR TASK DIFFERENCES")
print("=" * 80)

for pair in task_pairs:
    print(f"\n{pair.upper().replace('_', ' - ')}:")
    print("-" * 80)
    
    if 'thresholded' in diff_results[pair]:
        # Thresholded data
        thresh_data = diff_results[pair]['thresholded'].get_fdata()
        valid_thresh = thresh_data[~np.isnan(thresh_data)]
        
        if len(valid_thresh) > 0:
            print(f"  Thresholded (significant) differences:")
            print(f"    Number of significant voxels: {len(valid_thresh)}")
            print(f"    Mean difference: {np.mean(valid_thresh):.3f}")
            print(f"    Std difference: {np.std(valid_thresh):.3f}")
            print(f"    Range: [{np.min(valid_thresh):.3f}, {np.max(valid_thresh):.3f}]")
            
            # Count positive and negative differences
            n_pos = np.sum(valid_thresh > 0)
            n_neg = np.sum(valid_thresh < 0)
            print(f"    Positive differences (rest > movie): {n_pos} voxels")
            print(f"    Negative differences (rest < movie): {n_neg} voxels")
        else:
            print(f"  No significant differences found (after thresholding)")
    
    if 'unthresholded' in diff_results[pair]:
        # Unthresholded data for comparison
        unthresh_data = diff_results[pair]['unthresholded'].get_fdata()
        valid_unthresh = unthresh_data[~np.isnan(unthresh_data)]
        
        if len(valid_unthresh) > 0:
            print(f"\n  Unthresholded differences (all voxels):")
            print(f"    Number of voxels: {len(valid_unthresh)}")
            print(f"    Mean difference: {np.mean(valid_unthresh):.3f}")
            print(f"    Std difference: {np.std(valid_unthresh):.3f}")
            print(f"    Range: [{np.min(valid_unthresh):.3f}, {np.max(valid_unthresh):.3f}]")

print("\n" + "=" * 80)



## 5. ISC-IDE Correlation Analysis

Analyze the relationship between ISC and IDE within subjects for each movie task.



In [ ]:

from scipy import stats

# Run a correlation analysis between aeronaut IDE and aeronaut ISC, mickey IDE and mickey ISC within subjects

df_list = []

for task in ['aeronaut', 'mickey']:
    print(f"\n{'='*80}")
    print(f"Analyzing ISC-IDE correlation for {task.upper()}")
    print('='*80)
    
    # Load subject-level ISC and IDE data
    isc_fn = f'{results_dir}/{task}_ISC_all_subjects_results.nii.gz'
    ide_fn = f'{results_dir}/{task}_TPHATE_DiffOp_IDE_all_subjects_results.nii.gz'
    
    if os.path.exists(isc_fn) and os.path.exists(ide_fn):
        isc_img = nib.load(isc_fn)
        ide_img = nib.load(ide_fn)
        
        # Load the appropriate mask for this task
        mask = nib.load(arc.RM_INTERSECT_MASK)
        masker = NiftiMasker(mask_img=mask).fit(mask)
        
        # Apply the task-relevant intersection mask to both
        isc_data = masker.fit_transform(isc_img)
        ide_data = masker.fit_transform(ide_img)
        
        print(f"ISC data shape: {isc_data.shape}")
        print(f"IDE data shape: {ide_data.shape}")
        
        # Calculate subject-wise correlation
        # For each subject, correlate ISC values across voxels with IDE values across voxels
        n_subjects = isc_data.shape[0]
        results_map = np.zeros((n_subjects, 2))
        
        for i in range(n_subjects):
            # Get non-NaN voxels for this subject
            valid_mask = ~(np.isnan(isc_data[i, :]) | np.isnan(ide_data[i, :]))
            if np.sum(valid_mask) > 10:  # Need at least 10 voxels
                rho, pval = stats.spearmanr(isc_data[i, valid_mask], ide_data[i, valid_mask])
                results_map[i, :] = [rho, pval]
            else:
                results_map[i, :] = [np.nan, np.nan]
        
        temp = pd.DataFrame(results_map, columns=['spearman_rho', 'p_value'])
        temp['subject_id'] = np.arange(n_subjects)
        temp['task'] = task
        df_list.append(temp)
        
        # Print summary statistics
        valid_rhos = results_map[~np.isnan(results_map[:, 0]), 0]
        if len(valid_rhos) > 0:
            print(f"Mean correlation: {np.mean(valid_rhos):.3f} ± {np.std(valid_rhos):.3f}")
            print(f"Median correlation: {np.median(valid_rhos):.3f}")
    else:
        print(f"Warning: Could not find data files for {task}")

if df_list:
    res_df = pd.concat(df_list, ignore_index=True)
    print(f"\nCombined dataframe shape: {res_df.shape}")
else:
    print("Warning: No correlation results to combine")


In [ ]:
! ls adult_restmovie/results/aeronaut*ISC*atlas*

In [ ]:

from scipy import stats

# Run a correlation analysis between aeronaut IDE and aeronaut ISC, mickey IDE and mickey ISC within subjects

df_corr = pd.DataFrame(columns=['subject','rho','pval','zscore'])

task='aeronaut'
print(f"\n{'='*80}")
print(f"Analyzing ISC-IDE correlation for {task.upper()}")
print('='*80)

# Load subject-level ISC and IDE data
isc_data = np.load(f'{results_dir}/{task}_ISC_all_subject_results_atlas.npy')
ide_data = np.load(f'{results_dir}/{task}_TPHATE_DiffOp_IDE_all_subject_results_atlas.npy')
for sub in range(11):
    rho = np.corrcoef(isc_data[sub, :], ide_data[sub, :])[0, 1]
    # Compute zscore
    pval, obs, zsc = sh.permute_pattern(ide_data[sub, :], isc_data[sub, :], n_permutations=1000, random_state=4)
    df_corr.loc[len(df_corr)] = [sub, obs, pval, zsc
                            ]
df_corr.to_csv(f'{aru.get_results_dir()}/ISC_TPHATE_DiffOp_corr.csv', index=False)




In [ ]:
df_corr['zscore'].mean()

In [ ]:
import plotting_helpers

In [ ]:
# Plot these results as a bar plot with error bars
plt.figure(figsize=(4,6))
sns.barplot(data=res_df, x='task', y='spearman_rho',palette=plotting_helpers.get_paired_palette(), edgecolor='k', linewidth=2, alpha=0.8)
sns.stripplot(data=res_df, x='task', y='spearman_rho',palette=plotting_helpers.get_paired_palette(), edgecolor='k', linewidth=1, size=10, jitter=True)
# Run t-tests to compare against zero
for task in res_df['task'].unique():
    task_data = res_df[res_df['task'] == task]['spearman_rho'].dropna()
    t_stat, p_val = stats.ttest_1samp(task_data, 0)
    print(f"T-test for {task}: t={t_stat:.3f}, p={p_val:.4f}")  
# plot significance asterisks

y_max = res_df['spearman_rho'].max() 
for idx, task in enumerate(res_df['task'].unique()):
    task_data = res_df[res_df['task'] == task]['spearman_rho'].dropna()
    t_stat, p_val = stats.ttest_1samp(task_data, 0)
    if p_val < 0.001:
        sig = '***'
    elif p_val < 0.01:
        sig = '**'
    elif p_val < 0.05:
        sig = '*'
    else:
        sig = 'n.s.'
    plt.text(idx, 0.8, sig, ha='center', va='bottom', fontsize=16)

plt.ylim(y_min := res_df['spearman_rho'].min() - 0.1, y_max + 0.15)
plt.title('Brain-wide correlation between ISC and IDE', fontsize=16)
plt.ylabel('Spearman Correlation (rho)', fontsize=14)
plt.xlabel('Movie Task', fontsize=14)
plt.axhline(0, color='k', linestyle='--')
sns.despine()

In [ ]:
from scipy.stats import ttest_rel, ttest_ind
results_dir = aru.get_results_dir()
# Load subject-level IDE data for all three tasks and compute voxel-wise averages per subject
ide_subject_averages = []

for task in ['aeronaut', 'mickey', 'rest']:
    ide_fn = f'{results_dir}/{task}_TPHATE_DiffOp_IDE_all_subjects_results.nii.gz'
    
    if os.path.exists(ide_fn):
        ide_img = nib.load(ide_fn)
        ide_data = ide_img.get_fdata()
        
        # Get number of subjects (4th dimension)
        n_subjects = ide_data.shape[3]
        
        # For each subject, compute the mean IDE across all voxels
        for subj_idx in range(n_subjects):
            subj_data = ide_data[:, :, :, subj_idx]
            # Compute mean, ignoring NaN values
            mean_ide = np.nanmean(subj_data)
            
            ide_subject_averages.append({
                'task': task,
                'subject_id': subj_idx,
                'mean_IDE': mean_ide
            })
        
        print(f"Loaded {task}: {n_subjects} subjects")
    else:
        print(f"Warning: {ide_fn} not found")


In [ ]:
import stats_helpers
# Create DataFrame
ide_df = pd.DataFrame(ide_subject_averages)
print(f"\nCreated DataFrame with {len(ide_df)} rows")
print(ide_df.head())
tasks=['aeronaut', 'mickey', 'rest']
# Create barplot
cmap = helper.get_paired_palette()[:2]+helper.get_paired_palette()[4:]

plt.figure(figsize=(6,8))
sns.barplot(data=ide_df, x='task', y='mean_IDE', 
            palette=cmap, 
            edgecolor='k', linewidth=2, alpha=0.8)
sns.stripplot(data=ide_df, x='task', y='mean_IDE', 
              palette=cmap,
                size=12, alpha=1, jitter=True, edgecolor='k', linewidth=1)

# Add statistical comparisons

# Compare pairs of tasks
task_pairs_to_compare = [('aeronaut', 'mickey'), ('aeronaut', 'rest'), ('mickey', 'rest')]

y_max = ide_df['mean_IDE'].max()
y_min = ide_df['mean_IDE'].min()

for idx, (task1, task2) in enumerate(task_pairs_to_compare):
    data1 = ide_df[ide_df['task'] == task1]['mean_IDE'].values
    data2 = ide_df[ide_df['task'] == task2]['mean_IDE'].values
    
    # Use independent samples t-test since different tasks may have different subjects
    _, p_val, _ = stats_helpers.paired_difference_null_distribution(data2.reshape(-1,1), data1.reshape(-1,1), alternative='greater', n_perm=10000)
    p_val=p_val[0]
    print(f"\nT-test {task1} vs {task2}:  p={p_val:.2f}")

# Plot pairwise significance
for idx, (task1, task2) in enumerate(task_pairs_to_compare):
    data1 = ide_df[ide_df['task'] == task1]['mean_IDE'].values
    data2 = ide_df[ide_df['task'] == task2]['mean_IDE'].values
    
    t_stat, p_val = ttest_ind(data1, data2)
    
    # Determine significance level
    if p_val < 0.001:
        sig = '***'
    elif p_val < 0.01:
        sig = '**'
    elif p_val < 0.05:
        sig = '*'
    else:
        sig = 'n.s.'
    
    # Determine y position for the annotation
    y, h, col = y_max + 0.2 + idx*0.4, 0.05, 'k'
    x1, x2 = tasks.index(task1), tasks.index(task2)
    plt.plot([x1, x2], [y,  y], lw=1.5, c=col)
    plt.text((x1 + x2) * .5, y + h, sig, ha='center', va='bottom', color=col, fontsize=14)

plt.ylabel('Mean IDE (across voxels)', fontsize=14)
plt.xlabel('Task', fontsize=14)
plt.title('Average IDE by Task', fontsize=16)
sns.despine()
plt.tight_layout()


## 6. Composite plot: Top 10% Aeronaut ISC highlighted on Rest–Aeronaut thresholded difference map

This plot combines two pieces of information in a single brain visualization:
- Voxels where **Aeronaut ISC** is in the **top 10%** are highlighted in **yellow**.
- All other voxels show values from the **`aeronaut_rest` thresholded** (Rest − Aeronaut) IDE difference map.


In [ ]:
import numpy as np
import os
import nibabel as nib
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from nilearn import plotting

# --- Inputs ---
isc_img = avg_results.get('aeronaut', {}).get('ISC', None)
if isc_img is None:
    raise RuntimeError("Aeronaut ISC image not loaded. Ensure section 1 loaded 'aeronaut' ISC average results.")

if 'aeronaut_rest' not in diff_results or 'thresholded' not in diff_results['aeronaut_rest']:
    raise RuntimeError("aeronaut_rest thresholded difference map not loaded. Ensure section 3 loaded it.")

diff_img = diff_results['aeronaut_rest']['thresholded']

isc_data = isc_img.get_fdata()
diff_data = diff_img.get_fdata()

# Basic shape check
if isc_data.shape != diff_data.shape:
    raise ValueError(f"Shape mismatch: ISC {isc_data.shape} vs diff {diff_data.shape}")

# --- Top 10% ISC mask (ignore NaNs) ---
isc_valid = isc_data[~np.isnan(isc_data)]
if isc_valid.size == 0:
    raise RuntimeError("ISC image contains only NaNs.")
threshold=95
isc_thr = np.nanpercentile(isc_valid, threshold)
top_mask = (isc_data >= isc_thr) & ~np.isnan(isc_data)

print(f"Aeronaut ISC {threshold}th percentile threshold: {isc_thr:.4f}")
print(f"{threshold}% voxel count: {np.sum(top_mask)}")
# Get how many voxels show the significant difference



# --- Build composite volume ---
# We create a numeric code volume where:
#   2 -> top10 ISC voxels (to be colored yellow)
#   sign(diff) -> binarized by magnitude levels for a diverging map elsewhere
#
# For the thresholded diff map, zeros/NaNs are background; keep NaNs as background.
comp = np.full(diff_data.shape, np.nan, dtype=float)

# Non-NaN diff voxels that are not in top10 ISC -> keep diff values
keep_diff = (~np.isnan(diff_data)) & (~top_mask)
comp[keep_diff] = diff_data[keep_diff]

# Overwrite with a sentinel value for highlighted voxels (even if diff is NaN)
comp[top_mask] = 999.0

comp_img = nib.Nifti1Image(comp, affine=diff_img.affine, header=diff_img.header)


In [ ]:
np.sum(significant_diff_mask)

In [ ]:
threshold = 98  # top 5% ISC
isc_thr = np.nanpercentile(isc_valid, threshold)
top_mask = (isc_data >= isc_thr) & ~np.isnan(isc_data)
top_mask_vol = np.full(diff_img.shape, np.nan, dtype=float)
top_mask_vol[top_mask] = 1.0
top_mask_img = nib.Nifti1Image(top_mask_vol, affine=diff_img.affine, header=diff_img.header)
masker = NiftiMasker(mask_img=aru.get_intersect_mask())

significant_diff_mask = masker.fit().transform(diff_img) > 0  # Assuming positive differences are significant
top_isc_mask = masker.transform(top_mask_img) > 0
print(np.sum(significant_diff_mask), np.sum(top_isc_mask))
sh.overlap_coefficient(np.squeeze(significant_diff_mask), np.squeeze(top_isc_mask))


In [ ]:
# Layer 0: the thresholded Rest−Aeronaut IDE difference map (diverging colormap)
# Layer 1: a binary/top-percentile ISC mask (yellow overlay)
top_mask_vol = np.full(diff_img.shape, np.nan, dtype=float)
top_mask_vol[top_mask] = 1.0
top_mask_img = nib.Nifti1Image(top_mask_vol, affine=diff_img.affine, header=diff_img.header)

helper.generate_surface_plot(
    data_fn=[diff_img, top_mask_img],
    image_fn="main_plots/adult_aeronaut_rest_thresholded_with_top_ISC_overlay.pdf",  # saves to 
    atlas='searchlight',
    cmap=helper.diverging_colormap_bp(),
    cbar_range=(-11, 11),
    surf_type='fslr',
    target_density='32k',
    include_cbar=True,  # broadcast; overridden per-layer below
    title='Rest − Aeronaut IDE (thresholded) + Top ISC voxels',
    method='linear',
    threshold=None,
    alpha=0.5,
    mask_medial_wall=True,
    layers_kwargs=[
        dict(
            cmap=helper.diverging_colormap_bp(),
            cbar_range=(-11, 11),
            alpha=1,
            label='ΔIDE (Rest − Aeronaut)',
            cbar=True,
        ),
        dict(
            cmap=ListedColormap([(1,1,0.6,1.0)]), 
            cbar_range=(0, 1),
            alpha=0.8,
            label=f'Top {100 - threshold}% ISC mask',
            cbar=True,
        ),
    ],
)